## IsTheWorldReadyForTheNextPandemic - Mini Project
### Project handled by Liza, Ravit, Hagit and Hodaya

**This notebook includes preparation of the actual base dataset which trainig model will run upon.**

Remark: Data analysis will be done per Country_code[Location_key] and not per Country_code and regional

Comment: baseline data, excluding regional or sub-national slices



---
## Part 1 · Setup, data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Core imports.
import numpy as np
np.random.seed(42)  # For reproducibility
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set the path where your DataSet CSVs located.
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'


In [ ]:

# Load the standard/smaller files
country_index = pd.read_csv(data_dir + "country Index.csv")
ict_adoption = pd.read_csv(data_dir + "ict adoption by 100 people.csv")
ghs_index = pd.read_csv(data_dir + "2021-GHS-Index-April-2022.csv")
health = pd.read_csv(data_dir + "health.csv")
demographics = pd.read_csv(data_dir + "demographics.csv")
covid_daily = pd.read_csv(data_dir + "WHO-COVID-19-global-daily-data.csv")

print("Standard files loaded successfully!")

# Load the Big files - vaccinations/epidemiology
vaccinations = pd.read_csv(data_dir + "vaccinations.csv")
epidemiology = pd.read_csv(data_dir + "epidemiology.csv")

print("vaccinations/epidemiology BIG files loaded successfully!")


Standard files loaded successfully!
vaccinations/epidemiology BIG files loaded successfully!


# DataSet modification, preparation for model training
The purpose of this section is to create baseline dataset tables and modify/prepare them for model traning and science query by removing unnecessary data/remove null values etc

In [ ]:
# ==================================================
# Clean country Index.csv
# ==================================================

# Clean country Index.csv tables and leave only the records with aggregation_level equal to zero
# aggregation_level=1 means a district under a country
# aggregation_level=2 means a Sub-region within a district

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATH
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

input_filename = "country Index.csv"
output_filename = "Country_Ind_md.csv"

input_path = os.path.join(data_dir, input_filename)
output_path = os.path.join(data_dir, output_filename)

print("Initializing dataset modification pipeline...")

# ==========================================
# 2. UPLOAD & READ RAW SOURCE FILE
# ==========================================
if not os.path.exists(input_path):
    raise FileNotFoundError(f"❌ Error: Could not find the file at {input_path}. Please verify your Google Drive mounting and folder name.")

print(f"Reading source file: {input_filename}")
df_country = pd.read_csv(input_path)
print(f"-> Original Baseline Shape: {df_country.shape}")

# ==========================================
# 3. APPLY DATA MODIFICATION (FILTERING)
# ==========================================
print("Filtering records where aggregation_level == 0...")

# Standardize column naming just in case there are trailing spaces in the CSV header
df_country.columns = df_country.columns.str.strip()

# Create the modified slice containing only country-level rows - meaning aggregation_level is zero
df_country_md = df_country[df_country['aggregation_level'] == 0].copy()

print(f"-> Filtered Dataset Shape: {df_country_md.shape}")

# ==========================================
# 4. CREATE NEW PHYSICAL FILE IN GOOGLE DRIVE
# ==========================================
print(f"Writing new file to Drive: {output_filename}")

# Index=False prevents Pandas from adding a redundant, unnamed sequential column to your new CSV file
df_country_md.to_csv(output_path, index=False)

print("=" * 60)
print("✅ Success! Physical file created and saved securely.")
print(f"File Location: {output_path}")
print("=" * 60)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 5 ROWS OF THE MODIFIED COUNTRY INDEX FILTRATION ---")
pd.set_option('display.max_columns', 10)
print(df_country_md.head(5))


Initializing dataset modification pipeline...
Reading source file: country Index.csv
-> Original Baseline Shape: (22963, 8)
Filtering records where aggregation_level == 0...
-> Filtered Dataset Shape: (246, 8)
Writing new file to Drive: Country_Ind_md.csv
✅ Success! Physical file created and saved securely.
File Location: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/Country_Ind_md.csv

--- FIRST 5 ROWS OF THE MODIFIED COUNTRY INDEX FILTRATION ---
   location_key country_code          country_name subregion1_code  \
0            AD           AD               Andorra             NaN   
1            AE           AE  United Arab Emirates             NaN   
2            AF           AF           Afghanistan             NaN   
37           AG           AG   Antigua and Barbuda             NaN   
38           AI           AI              Anguilla             NaN   

   subregion1_name subregion2_code subregion2_name  aggregation_level  
0              NaN        

In [ ]:

# =======================================================================
# Merge "Country_Ind_md.csv" and "demographics.csv" per Location_key
# =======================================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

country_file = "Country_Ind_md.csv"
demo_file = "demographics.csv"
output_file = "demographics_Ind_md.csv"

path_country = os.path.join(data_dir, country_file)
path_demo = os.path.join(data_dir, demo_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing cross-table integration pipeline...")

# ==========================================
# 2. UPLOAD & READ BOTH FILES
# ==========================================
if not os.path.exists(path_country) or not os.path.exists(path_demo):
    raise FileNotFoundError("❌ Error: Missing source files. Please ensure both 'Country_Ind_md.csv' and 'demographics.csv' are present in the directory.")

print(f"Reading: {country_file}")
df_country_md = pd.read_csv(path_country)

print(f"Reading: {demo_file}")
df_demographics = pd.read_csv(path_demo)

# Clean and strip column names to ensure perfect matching mechanics
df_country_md.columns = df_country_md.columns.str.strip()
df_demographics.columns = df_demographics.columns.str.strip()

print(f"-> Country Index (Modified) Shape: {df_country_md.shape}")
print(f"-> Demographics Baseline Shape:   {df_demographics.shape}\n")

# ==========================================
# 3. PERFORM INNER JOIN USING 'location_key'
# ==========================================
print("Merging datasets on matching 'location_key' codes...")

df_merged_profile = pd.merge(
    df_country_md,
    df_demographics,
    on='location_key',
    how='inner'
)

print(f"-> Integrated Profile Shape:      {df_merged_profile.shape}")

# ==========================================
# 4. EXPORT INTEGRATED DATA TO PHYSICAL CSV
# ==========================================
print(f"Writing integrated file to Drive: {output_file}")

# index=False ensures no unnecessary, unnamed tracking column is created in your file
df_merged_profile.to_csv(path_output, index=False)

print("=" * 65)
print("✅ Success! Integrated demographics file created and saved.")
print(f"Target Destination: {path_output}")
print("=" * 65)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MERGED DEMOGRAPHICS PROFILE ---")
pd.set_option('display.max_columns', 12)
print(df_merged_profile.head(10))

Initializing cross-table integration pipeline...
Reading: Country_Ind_md.csv
Reading: demographics.csv
-> Country Index (Modified) Shape: (246, 8)
-> Demographics Baseline Shape:   (21689, 19)

Merging datasets on matching 'location_key' codes...
-> Integrated Profile Shape:      (246, 26)
Writing integrated file to Drive: demographics_Ind_md.csv
✅ Success! Integrated demographics file created and saved.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/demographics_Ind_md.csv

--- FIRST 10 ROWS OF THE MERGED DEMOGRAPHICS PROFILE ---
  location_key country_code          country_name  subregion1_code  \
0           AD           AD               Andorra              NaN   
1           AE           AE  United Arab Emirates              NaN   
2           AF           AF           Afghanistan              NaN   
3           AG           AG   Antigua and Barbuda              NaN   
4           AI           AI              Anguilla              Na

In [ ]:
# =======================================================================
# Merge "Country_Ind_md.csv" and "health.csv" per Location_key
# =======================================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

country_file = "Country_Ind_md.csv"
health_file = "health.csv"
output_file = "health_Ind_md.csv"

path_country = os.path.join(data_dir, country_file)
path_health = os.path.join(data_dir, health_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing health infrastructure integration pipeline...")

# ==========================================
# 2. UPLOAD & READ BOTH FILES
# ==========================================
if not os.path.exists(path_country) or not os.path.exists(path_health):
    raise FileNotFoundError("❌ Error: Missing source files. Please ensure both 'Country_Ind_md.csv' and 'health.csv' are present in the directory.")

print(f"Reading: {country_file}")
df_country_md = pd.read_csv(path_country)

print(f"Reading: {health_file}")
df_health = pd.read_csv(path_health)

# Clean and strip column names to ensure perfect matching mechanics
df_country_md.columns = df_country_md.columns.str.strip()
df_health.columns = df_health.columns.str.strip()

print(f"-> Country Index (Modified) Shape: {df_country_md.shape}")
print(f"-> Health Baseline Shape:          {df_health.shape}\n")

# ==========================================
# 3. PERFORM INNER JOIN USING 'location_key'
# ==========================================
print("Merging datasets on matching 'location_key' codes...")

df_merged_health = pd.merge(
    df_country_md,
    df_health,
    on='location_key',
    how='inner'
)

print(f"-> Integrated Health Shape:        {df_merged_health.shape}")

# ==========================================
# 4. EXPORT INTEGRATED DATA TO PHYSICAL CSV
# ==========================================
print(f"Writing integrated file to Drive: {output_file}")

# index=False ensures no unnecessary, unnamed tracking column is created in your file
df_merged_health.to_csv(path_output, index=False)

print("=" * 65)
print("✅ Success! Integrated health infrastructure file created and saved.")
print(f"Target Destination: {path_output}")
print("=" * 65)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MERGED HEALTH PROFILE ---")
pd.set_option('display.max_columns', 12)
print(df_merged_health.head(10))

Initializing health infrastructure integration pipeline...
Reading: Country_Ind_md.csv
Reading: health.csv
-> Country Index (Modified) Shape: (246, 8)
-> Health Baseline Shape:          (3504, 14)

Merging datasets on matching 'location_key' codes...
-> Integrated Health Shape:        (210, 21)
Writing integrated file to Drive: health_Ind_md.csv
✅ Success! Integrated health infrastructure file created and saved.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/health_Ind_md.csv

--- FIRST 10 ROWS OF THE MERGED HEALTH PROFILE ---
  location_key country_code          country_name  subregion1_code  \
0           AD           AD               Andorra              NaN   
1           AE           AE  United Arab Emirates              NaN   
2           AF           AF           Afghanistan              NaN   
3           AG           AG   Antigua and Barbuda              NaN   
4           AL           AL               Albania              NaN   

In [ ]:

# ==================================================
# Update "WHO-COVID-19-global-daily-data.csv"
# ==================================================

# Modify "WHO-COVID-19-global-daily-data.csv" table as follow
# New_cases=0 where field content is null/empty
# New_deaths=0 where field content is null/empty

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

input_file = "WHO-COVID-19-global-daily-data.csv"
output_file = "WHO-COVID-19-global-daily-data_md.csv"

path_input = os.path.join(data_dir, input_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing missing value cleanup pipeline...")

# ==========================================
# 2. UPLOAD & READ RAW SOURCE FILE
# ==========================================
if not os.path.exists(path_input):
    raise FileNotFoundError(f"❌ Error: Could not find the source file at {path_input}. Please check the folder connection.")

print(f"Reading: {input_file}")
df_covid = pd.read_csv(path_input)

# Clean up column spaces to guarantee strict target matching
df_covid.columns = df_covid.columns.str.strip()

print(f"-> Base Dataset Shape: {df_covid.shape}")

# ==========================================
# 3. AUDIT & IMPUTE MISSING VALUES WITH 0
# ==========================================
target_cols = ['New_cases', 'New_deaths']

print("\n--- NULL VALUE AUDIT BEFORE CLEANUP ---")
for col in target_cols:
    if col in df_covid.columns:
        null_count = df_covid[col].isnull().sum()
        print(f"Column '{col}': found {null_count} missing values.")
    else:
        print(f"⚠️ Warning: Column '{col}' not found in dataset columns!")

print("\nExecuting data modification (Filling Nulls with 0)...")
# Safely replace all missing values in our target parameters with the actual integer 0
for col in target_cols:
    if col in df_covid.columns:
        df_covid[col] = df_covid[col].fillna(0)

print("\n--- NULL VALUE AUDIT AFTER CLEANUP ---")
for col in target_cols:
    if col in df_covid.columns:
        null_count = df_covid[col].isnull().sum()
        print(f"Column '{col}': found {null_count} missing values.")

# ==========================================
# 4. EXPORT IMMUTABLE CLEANED DATA TO CSV
# ==========================================
print(f"\nWriting modified dataset to Drive: {output_file}")

# index=False drops the default sequential tracking axis from being permanently baked into the file
df_covid.to_csv(path_output, index=False)

print("=" * 70)
print("✅ Success! Modified COVID-19 dataset created and saved securely.")
print(f"Target Destination: {path_output}")
print("=" * 70)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MODIFIED TIMELINE PROFILE ---")
pd.set_option('display.max_columns', 12)
print(df_covid.head(10))



Initializing missing value cleanup pipeline...
Reading: WHO-COVID-19-global-daily-data.csv
-> Base Dataset Shape: (558240, 8)

--- NULL VALUE AUDIT BEFORE CLEANUP ---
Column 'New_cases': found 344261 missing values.
Column 'New_deaths': found 402613 missing values.

Executing data modification (Filling Nulls with 0)...

--- NULL VALUE AUDIT AFTER CLEANUP ---
Column 'New_cases': found 0 missing values.
Column 'New_deaths': found 0 missing values.

Writing modified dataset to Drive: WHO-COVID-19-global-daily-data_md.csv
✅ Success! Modified COVID-19 dataset created and saved securely.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/WHO-COVID-19-global-daily-data_md.csv

--- FIRST 10 ROWS OF THE MODIFIED TIMELINE PROFILE ---
  Date_reported Country_code              Country WHO_region  New_cases  \
0    04/01/2020           AF          Afghanistan        EMR        0.0   
1    04/01/2020           DZ              Algeria        AFR        0.0 

In [ ]:
# =======================================================================
# Merge "Country_Ind_md.csv" and "vaccination.csv" per Location_key
# =======================================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

country_file = "Country_Ind_md.csv"
vaccination_file = "vaccinations.csv"
output_file = "vaccinations_Ind_md.csv"

path_country = os.path.join(data_dir, country_file)
path_vac = os.path.join(data_dir, vaccination_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing vaccination timeline integration pipeline...")

# ==========================================
# 2. UPLOAD & READ BOTH FILES
# ==========================================
if not os.path.exists(path_country) or not os.path.exists(path_vac):
    raise FileNotFoundError("❌ Error: Missing source files. Please ensure both 'Country_Ind_md.csv' and 'vaccinations.csv' are present in the directory.")

print(f"Reading: {country_file}")
df_country_md = pd.read_csv(path_country)

print(f"Reading: {vaccination_file}")
df_vac = pd.read_csv(path_vac)

# Clean and strip column names to ensure perfect matching mechanics
df_country_md.columns = df_country_md.columns.str.strip()
df_vac.columns = df_vac.columns.str.strip()

print(f"-> Country Index (Modified) Shape: {df_country_md.shape}")
print(f"-> Vaccinations Baseline Shape:    {df_vac.shape}\n")

# ==========================================
# 3. PERFORM INNER JOIN USING 'location_key'
# ==========================================
print("Merging datasets on matching 'location_key' codes...")

df_merged_vac = pd.merge(
    df_country_md,
    df_vac,
    on='location_key',
    how='inner'
)

print(f"-> Integrated Vaccination Shape:   {df_merged_vac.shape}")

# ==========================================
# 4. EXPORT INTEGRATED DATA TO PHYSICAL CSV
# ==========================================
print(f"Writing integrated file to Drive: {output_file}")

# index=False ensures no unnecessary, unnamed tracking column is created in your file
df_merged_vac.to_csv(path_output, index=False)

print("=" * 65)
print("✅ Success! Integrated vaccination timeline file created and saved.")
print(f"Target Destination: {path_output}")
print("=" * 65)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MERGED VACCINATION PROFILE ---")
pd.set_option('display.max_columns', 12)
print(df_merged_vac.head(10))


Initializing vaccination timeline integration pipeline...
Reading: Country_Ind_md.csv
Reading: vaccinations.csv
-> Country Index (Modified) Shape: (246, 8)
-> Vaccinations Baseline Shape:    (2545118, 32)

Merging datasets on matching 'location_key' codes...
-> Integrated Vaccination Shape:   (56031, 39)
Writing integrated file to Drive: vaccinations_Ind_md.csv
✅ Success! Integrated vaccination timeline file created and saved.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/vaccinations_Ind_md.csv

--- FIRST 10 ROWS OF THE MERGED VACCINATION PROFILE ---
  location_key country_code country_name  subregion1_code  subregion1_name  \
0           AD           AD      Andorra              NaN              NaN   
1           AD           AD      Andorra              NaN              NaN   
2           AD           AD      Andorra              NaN              NaN   
3           AD           AD      Andorra              NaN              NaN   
4  

In [ ]:
# =======================================================================
# Merge "Country_Ind_md.csv" and "epidemiology.csv" per Location_key
# =======================================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

country_file = "Country_Ind_md.csv"
epi_file = "epidemiology.csv"
output_file = "epidemiology_Ind_md.csv"

path_country = os.path.join(data_dir, country_file)
path_epi = os.path.join(data_dir, epi_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing epidemiology timeline integration pipeline...")

# ==========================================
# 2. UPLOAD & READ BOTH FILES
# ==========================================
if not os.path.exists(path_country) or not os.path.exists(path_epi):
    raise FileNotFoundError("❌ Error: Missing source files. Please ensure both 'Country_Ind_md.csv' and 'epidemiology.csv' are present in the directory.")

print(f"Reading: {country_file}")
df_country_md = pd.read_csv(path_country)

print(f"Reading: {epi_file}")
df_epi = pd.read_csv(path_epi)

# Clean and strip column names to ensure perfect matching mechanics
df_country_md.columns = df_country_md.columns.str.strip()
df_epi.columns = df_epi.columns.str.strip()

print(f"-> Country Index (Modified) Shape: {df_country_md.shape}")
print(f"-> Epidemiology Baseline Shape:    {df_epi.shape}\n")

# ==========================================
# 3. PERFORM INNER JOIN USING 'location_key'
# ==========================================
print("Merging datasets on matching 'location_key' codes...")

df_merged_epi = pd.merge(
    df_country_md,
    df_epi,
    on='location_key',
    how='inner'
)

print(f"-> Integrated Epidemiology Shape:   {df_merged_epi.shape}")

# ==========================================
# 4. EXPORT INTEGRATED DATA TO PHYSICAL CSV
# ==========================================
print(f"Writing integrated file to Drive: {output_file}")

# index=False ensures no unnecessary, unnamed tracking column is created in your file
df_merged_epi.to_csv(path_output, index=False)

print("=" * 65)
print("✅ Success! Integrated epidemiology timeline file created and saved.")
print(f"Target Destination: {path_output}")
print("=" * 65)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MERGED EPIDEMIOLOGY PROFILE ---")
pd.set_option('display.max_columns', 12)
print(df_merged_epi.head(10))


Initializing epidemiology timeline integration pipeline...
Reading: Country_Ind_md.csv
Reading: epidemiology.csv
-> Country Index (Modified) Shape: (246, 8)
-> Epidemiology Baseline Shape:    (12525825, 10)

Merging datasets on matching 'location_key' codes...
-> Integrated Epidemiology Shape:   (227879, 17)
Writing integrated file to Drive: epidemiology_Ind_md.csv
✅ Success! Integrated epidemiology timeline file created and saved.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/epidemiology_Ind_md.csv

--- FIRST 10 ROWS OF THE MERGED EPIDEMIOLOGY PROFILE ---
  location_key country_code country_name  subregion1_code  subregion1_name  \
0           AD           AD      Andorra              NaN              NaN   
1           AD           AD      Andorra              NaN              NaN   
2           AD           AD      Andorra              NaN              NaN   
3           AD           AD      Andorra              NaN              NaN 

In [ ]:
# =======================================================================
# Validate "2021-GHS-Index-April-2022.csv" has no null values inside
# =======================================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATH
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
input_file = "2021-GHS-Index-April-2022.csv"
path_input = os.path.join(data_dir, input_file)

print("Initializing GHS Index null value audit pipeline...")

# ==========================================
# 2. UPLOAD & READ RAW SOURCE FILE
# ==========================================
if not os.path.exists(path_input):
    raise FileNotFoundError(f"❌ Error: Could not find the file at {path_input}. Please verify your Google Drive connection.")

df_ghs = pd.read_csv(path_input)
print(f"-> Dataset successfully loaded. Shape: {df_ghs.shape}\n")

# ==========================================
# 3. COMPUTE AND EVALUATE NULL FIELDS
# ==========================================
# Calculate the total number of missing values across every field
null_counts = df_ghs.isnull().sum()
total_nulls = null_counts.sum()

print("=" * 65)
print("                  NULL VALUE AUDIT REPORT")
print("=" * 65)

# Check if any missing values exist in the entire matrix
if total_nulls > 0:
    print(f"⚠️ Audit State: Found a total of {total_nulls} null/empty cells in the table.\n")
    print("Columns containing missing values and their respective counts:")

    # Isolate and display only fields that contain 1 or more missing values
    columns_with_nulls = null_counts[null_counts > 0]
    print(columns_with_nulls)
else:
    # Print the exact required confirmation message if no nulls are found
    print("File is ready to be used - no null/empty value inside")

print("=" * 65)

Initializing GHS Index null value audit pipeline...
-> Dataset successfully loaded. Shape: (390, 313)

                  NULL VALUE AUDIT REPORT
File is ready to be used - no null/empty value inside


In [ ]:
# =======================================================================
# Merge "2021-GHS-Index-April-2022.csv" and "Country_Ind_md.csv"
# =======================================================================

# Country_name is the field combine between the two tables by inner join

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

country_file = "Country_Ind_md.csv"
ghs_file = "2021-GHS-Index-April-2022.csv"
output_file = "2021-GHS-Index-April-2022_Ind_md.csv"

path_country = os.path.join(data_dir, country_file)
path_ghs = os.path.join(data_dir, ghs_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing GHS Index integration pipeline...")

# ==========================================
# 2. UPLOAD & READ BOTH FILES
# ==========================================
if not os.path.exists(path_country) or not os.path.exists(path_ghs):
    raise FileNotFoundError("❌ Error: Missing source files. Please ensure both 'Country_Ind_md.csv' and '2021-GHS-Index-April-2022.csv' are present in the directory.")

print(f"Reading: {country_file}")
df_country_md = pd.read_csv(path_country)

print(f"Reading: {ghs_file}")
df_ghs = pd.read_csv(path_ghs)

# Clean and strip column names to ensure perfect matching mechanics
df_country_md.columns = df_country_md.columns.str.strip()
df_ghs.columns = df_ghs.columns.str.strip()

print(f"-> Country Index (Modified) Shape: {df_country_md.shape}")
print(f"-> GHS Index Baseline Shape:       {df_ghs.shape}\n")

# ==========================================
# 3. STANDARDIZE & PERFORM INNER JOIN
# ==========================================
print("Standardizing match identifiers and merging datasets...")

# Ensure casing and trailing spaces don't interfere with the text match
df_country_md['match_country'] = df_country_md['country_name'].astype(str).str.strip().str.lower()
df_ghs['match_country'] = df_ghs['Country'].astype(str).str.strip().str.lower()

df_merged_ghs = pd.merge(
    df_country_md,
    df_ghs,
    on='match_country',
    how='inner'
)

# Clean up the temporary alignment columns used for joining
df_merged_ghs = df_merged_ghs.drop(columns=['match_country'])

print(f"-> Integrated GHS Index Shape:     {df_merged_ghs.shape}")

# ==========================================
# 4. EXPORT INTEGRATED DATA TO PHYSICAL CSV
# ==========================================
print(f"Writing integrated file to Drive: {output_file}")

# index=False ensures no unnecessary, unnamed tracking column is created in your file
df_merged_ghs.to_csv(path_output, index=False)

print("=" * 65)
print("✅ Success! Integrated GHS Index file created and saved.")
print(f"Target Destination: {path_output}")
print("=" * 65)

# ==========================================
# 5. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MERGED GHS PROFILE ---")
pd.set_option('display.max_columns', 12)
print(df_merged_ghs.head(10))


Initializing GHS Index integration pipeline...
Reading: Country_Ind_md.csv
Reading: 2021-GHS-Index-April-2022.csv
-> Country Index (Modified) Shape: (246, 8)
-> GHS Index Baseline Shape:       (390, 313)

Standardizing match identifiers and merging datasets...
-> Integrated GHS Index Shape:     (362, 321)
Writing integrated file to Drive: 2021-GHS-Index-April-2022_Ind_md.csv
✅ Success! Integrated GHS Index file created and saved.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/2021-GHS-Index-April-2022_Ind_md.csv

--- FIRST 10 ROWS OF THE MERGED GHS PROFILE ---
  location_key country_code          country_name  subregion1_code  \
0           AD           AD               Andorra              NaN   
1           AD           AD               Andorra              NaN   
2           AE           AE  United Arab Emirates              NaN   
3           AE           AE  United Arab Emirates              NaN   
4           AF           AF        

In [ ]:
# ==================================================================================
# Merge "2021-GHS-Index-April-2022_Ind_md.csv" and "ict adoption by 100 people.csv"
# ==================================================================================

# Country/Country_name is the field combine between the two tables by inner join
# Before Merge activity both tables are clean and left with records of only Year = 2019 or 2021 and kept in temp tables

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'

ict_file = "ict adoption by 100 people.csv"
ghs_file = "2021-GHS-Index-April-2022_Ind_md.csv"
output_file = "ICT_GHS_Ind_md.csv"

path_ict = os.path.join(data_dir, ict_file)
path_ghs = os.path.join(data_dir, ghs_file)
path_output = os.path.join(data_dir, output_file)

print("Initializing ICT & GHS Year-Filtered Integration Pipeline...")

# ==========================================
# 2. READ SOURCE DATASETS
# ==========================================
if not os.path.exists(path_ict) or not os.path.exists(path_ghs):
    raise FileNotFoundError("❌ Error: Missing source files. Please ensure both files exist in your Drive directory.")

df_ict_raw = pd.read_csv(path_ict)
df_ghs_raw = pd.read_csv(path_ghs)

# Clean column headers
df_ict_raw.columns = df_ict_raw.columns.str.strip()
df_ghs_raw.columns = df_ghs_raw.columns.str.strip()

print(f"-> Raw ICT Adoption Table Shape:       {df_ict_raw.shape}")
print(f"-> Raw GHS Index Modified Table Shape: {df_ghs_raw.shape}\n")

# Ensure a Year column exists in the GHS table for filtering (adds it if missing, as it represents 2021 data)
if 'Year' not in df_ghs_raw.columns:
    df_ghs_raw['Year'] = 2021

# ==========================================
# 3. FILTER BOTH TABLES FOR YEARS 2019 OR 2021
# ==========================================
print("Filtering rows for Year 2019 or 2021 into temporary tables...")

# Step 1: Filter ICT Table
df_ict_temp = df_ict_raw[df_ict_raw['Year'].isin([2019, 2021])].copy()

# Step 2: Filter GHS Table
df_ghs_temp = df_ghs_raw[df_ghs_raw['Year'].isin([2019, 2021])].copy()

print(f"   - Temp ICT Table Shape (2019/2021): {df_ict_temp.shape}")
print(f"   - Temp GHS Table Shape (2019/2021): {df_ghs_temp.shape}\n")

# ==========================================
# 4. STANDARDIZE KEYS AND MERGE
# ==========================================
print("Standardizing match identifiers and merging data...")

# Determine the country name column in ICT table (handles 'Country' or 'Country Name')
ict_country_col = 'Entity' if 'Entity' in df_ict_temp.columns else 'Country'
ghs_country_col = 'Country' if 'Country' in df_ghs_temp.columns else 'country_name'

# Create standardized lowercase matching columns to safeguard against spacing differences
df_ict_temp['match_country'] = df_ict_temp[ict_country_col].astype(str).str.strip().str.lower()
df_ghs_temp['match_country'] = df_ghs_temp[ghs_country_col].astype(str).str.strip().str.lower()

# Inner join on BOTH Country and Year matching criteria
df_merged = pd.merge(
    df_ict_temp,
    df_ghs_temp,
    on=['match_country', 'Year'],
    how='inner'
)

# Remove the temporary match keys
df_merged = df_merged.drop(columns=['match_country'])

print(f"-> Final Filtered & Merged Table Shape: {df_merged.shape}")

# ==========================================
# 5. EXPORT INTEGRATED DATA TO PHYSICAL CSV
# ==========================================
print(f"Writing output file to Drive: {output_file}")
df_merged.to_csv(path_output, index=False)

print("=" * 75)
print("✅ Success! Filtered merge completed successfully.")
print(f"Target Destination: {path_output}")
print("=" * 75)

# ==========================================
# 6. INTEGRITY VALIDATION PREVIEW
# ==========================================
print("\n--- FIRST 10 ROWS OF THE MERGED DATASET ---")
pd.set_option('display.max_columns', None)
print(df_merged.head(10))

Initializing ICT & GHS Year-Filtered Integration Pipeline...
-> Raw ICT Adoption Table Shape:       (13232, 7)
-> Raw GHS Index Modified Table Shape: (362, 321)

Filtering rows for Year 2019 or 2021 into temporary tables...
   - Temp ICT Table Shape (2019/2021): (450, 7)
   - Temp GHS Table Shape (2019/2021): (362, 321)

Standardizing match identifiers and merging data...
-> Final Filtered & Merged Table Shape: (352, 327)
Writing output file to Drive: ICT_GHS_Ind_md.csv
✅ Success! Filtered merge completed successfully.
Target Destination: /content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/ICT_GHS_Ind_md.csv

--- FIRST 10 ROWS OF THE MERGED DATASET ---
        Entity Code  Year  Landline phone subscriptions  \
0  Afghanistan  AFG  2019                         0.356   
1  Afghanistan  AFG  2021                         0.364   
2      Albania  ALB  2019                         8.420   
3      Albania  ALB  2021                         6.940   
4      Algeria  DZA  

## Descriptive Statistic on new Base Dataset
Here we are running Descriptive statistic on each new given table of the dataset using describe() pandas function.

**In Pandas, describe()** is a built-in function that automatically computes a comprehensive summary of descriptive statistics for a DataFrame. Instead of writing separate code for the mean, median, min, max, and standard deviation, calling this single function calculates them all at once.

***By default***, it analyzes numerical columns and provides:
*   **count**: Number of non-missing (non-NaN) values.
*   **mean**: The arithmetic average.
*   **std**: Standard deviation (measures how spread out the data is).
*   **min**: The lowest value in the column.
*   **5%, 50%, 75%**: The percentiles (where 50% is the Median).
*   **max**: The highest value in the column.

To capture categorical (text) columns as well, we can use df.describe(include='all'), which adds metrics like unique (count of distinct values) and top (the most frequent value).

In [ ]:
# ==================================================
# 1. Descriptive Statistic - "Country_Ind_md.csv"
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "Country_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Setting options so Colab doesn't truncate the columns visually in the output window
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' forces Pandas to combine text and numeric summaries into a single unified table
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading Country_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: Country_Ind_md.csv         
       location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level
count           245          245          246              0.0              0.0              0.0              0.0              246.0
unique          245          245          246              NaN              NaN              NaN              NaN                NaN
top              AD           AD      Andorra              NaN              NaN              NaN              NaN                NaN
freq              1            1            1              NaN              NaN              NaN              NaN                NaN
mean            NaN          NaN          NaN              NaN              NaN              NaN              NaN                0.0
std             NaN          NaN          NaN              NaN              NaN 

In [ ]:
"""
This table contains static country healthcare metrics and pre-existing medical vulnerabilities (like life expectancy, smoking prevalence, diabetes prevalence, and physician counts).
These features will act as your main baseline risk predictors.
"""

# ==================================================
# 2. Descriptive Statistic - health_Ind_md.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "health_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading health_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: health_Ind_md.csv         
       location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level  life_expectancy  smoking_prevalence  diabetes_prevalence  infant_mortality_rate  adult_male_mortality_rate  adult_female_mortality_rate  pollution_mortality_rate  comorbidity_mortality_rate  hospital_beds_per_1000  nurses_per_1000  physicians_per_1000  health_expenditure_usd  out_of_pocket_health_expenditure_usd
count           209          209          210              0.0              0.0              0.0              0.0              210.0       205.000000          146.000000           209.000000             193.000000                 189.000000                   189.000000                183.000000                  183.000000               25.000000       180.000000           164.000000              186.000000                       

In [ ]:
"""
This table contains static structural population distributions across age brackets, density metrics, and development index indicators, which are highly critical for evaluating regional vulnerabilities.
"""

# ==================================================
# 3. Descriptive Statistic - demographics_Ind_md.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "demographics_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring options so Colab displays all data columns without text wrapping truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' forces a single matrix showing categorical distributions alongside numerical metrics
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading demographics_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: demographics_Ind_md.csv         
       location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level    population  population_male  population_female  population_rural  population_urban  population_largest_city  population_clustered  population_density  human_development_index  population_age_00_09  population_age_10_19  population_age_20_29  population_age_30_39  population_age_40_49  population_age_50_59  population_age_60_69  population_age_70_79  population_age_80_and_older
count           245          245          246              0.0              0.0              0.0              0.0              246.0  2.450000e+02     2.350000e+02       2.350000e+02      2.130000e+02      2.130000e+02             1.520000e+02          1.210000e+02          230.000000               186.000000          2.350000e+02          2.350000e+

In [ ]:
"""
This table contains the core global daily tracking indicators for cases and deaths reported directly to the WHO.
"""

# ===============================================================
# 4. Descriptive Statistic - WHO-COVID-19-global-daily-data_md.csv
# ===============================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "WHO-COVID-19-global-daily-data_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading WHO-COVID-19-global-daily-data_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: WHO-COVID-19-global-daily-data_md.csv         
       Date_reported Country_code      Country WHO_region     New_cases  Cumulative_cases     New_deaths  Cumulative_deaths
count         558240       555914       558240     558240  5.582400e+05      5.582400e+05  558240.000000       5.582400e+05
unique          2326          239          240          7           NaN               NaN            NaN                NaN
top       17/05/2026           AF  Afghanistan        EUR           NaN               NaN            NaN                NaN
freq             240         2326         2326     144212           NaN               NaN            NaN                NaN
mean             NaN          NaN          NaN        NaN  1.395860e+03      2.192818e+06      12.745717       2.265620e+04
std              NaN          NaN          NaN        NaN  2.986467e+04      8.918350e+06     12

In [ ]:
"""
This dataset tracks a nation's digital infrastructure capacity over time (mobile subscriptions, internet penetration, etc.), which will allow us to assess how well
technology aided communication and rapid response during the crisis.
"""

# ==================================================
# 5. Descriptive Statistic - ict adoption by 100 people.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "ict adoption by 100 people.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading ict adoption by 100 people.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: ict adoption by 100 people.csv         
                              Entity    Code          Year  Landline phone subscriptions  Landline Internet subscriptions  Mobile phone subscriptions  Internet users
count                          13232   13232  13232.000000                  12432.000000                      4428.000000                11064.000000     6331.000000
unique                           227     227           NaN                           NaN                              NaN                         NaN             NaN
top     Europe and Central Asia (WB)  WB_ECA           NaN                           NaN                              NaN                         NaN             NaN
freq                              66      66           NaN                           NaN                              NaN                         NaN             NaN
mean                    

In [ ]:
"""
This table contains the Global Health Security Index dimensions, evaluating countries across critical health security pillars such as early detection, rapid response, compliance
with international norms, and risk environments.
"""

# ==================================================
# 6. Descriptive Statistic - 2021-GHS-Index-April-2022_Ind_md.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "2021-GHS-Index-April-2022_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)



Loading 2021-GHS-Index-April-2022_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: 2021-GHS-Index-April-2022_Ind_md.csv         
       location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level  Country         Year  OVERALL SCORE  1) PREVENTION OF THE EMERGENCE OR RELEASE OF PATHOGENS  1.1) Antimicrobial resistance (AMR)  1.1.1) AMR surveillance, detection and reporting  1.1.1a) National plan for AMR priority pathogens  1.1.1b) Capacity of national lab/lab system to test for AMR priority pathogens  1.1.1c) National environmental surveillance for AMR residues/organisms  1.1.2) Antimicrobial control  1.1.2a) National law(s) requiring prescription for antibiotic use (humans)  1.1.2b) National law(s) requiring prescription for antibiotic use (animals)  1.2) Zoonotic disease  1.2.1) National planning for zoonotic diseases/pathogens  1.2.1a) Laws/plans on zoonotic disease  1.2.1b) Laws/plans on 

In [ ]:
"""
This is one of the heavy time-series tracking files, containing day-by-day records of confirmed cases, active cases, recoveries, and mortality figures grouped by geographic location identifiers.
"""

# ==================================================
# 7. Descriptive Statistic - epidemiology_Ind_md.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "epidemiology_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading epidemiology_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: epidemiology_Ind_md.csv         
       location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level        date  new_confirmed   new_deceased  new_recovered    new_tested  cumulative_confirmed  cumulative_deceased  cumulative_recovered  cumulative_tested
count        226892       226892       227879              0.0              0.0              0.0              0.0           227879.0      227879   2.277860e+05  227611.000000   1.571200e+04  7.727000e+04          2.277850e+05         2.276120e+05          1.475900e+04       8.065700e+04
unique          232          232          233              NaN              NaN              NaN              NaN                NaN         991            NaN            NaN            NaN           NaN                   NaN                  NaN                   NaN                NaN
top 

In [ ]:
"""
This table contains daily metrics on vaccine doses administered, cumulative rollout volumes, and
the count of persons who have completed their vaccination schedule per region.
"""

# ==================================================
# 8. Descriptive Statistic - vaccinations_Ind_md.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "vaccinations_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)

Loading vaccinations_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: vaccinations_Ind_md.csv         
       location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level        date  new_persons_vaccinated  cumulative_persons_vaccinated  new_persons_fully_vaccinated  cumulative_persons_fully_vaccinated  new_vaccine_doses_administered  cumulative_vaccine_doses_administered  new_persons_vaccinated_pfizer  cumulative_persons_vaccinated_pfizer  new_persons_fully_vaccinated_pfizer  cumulative_persons_fully_vaccinated_pfizer  new_vaccine_doses_administered_pfizer  cumulative_vaccine_doses_administered_pfizer  new_persons_vaccinated_moderna  cumulative_persons_vaccinated_moderna  new_persons_fully_vaccinated_moderna  cumulative_persons_fully_vaccinated_moderna  new_vaccine_doses_administered_moderna  cumulative_vaccine_doses_administered_moderna  new_persons_vaccinated_janssen  cumulative_persons_va

In [ ]:
"""
This table is the output results of merge activity between ict adoption by 100 people.csv and 2021-GHS-Index-April-2022_Ind_md.csv table
where first each of those tables were cleaned and remains with records of Year=2019 or 2021 only
 """

# ==================================================
# 8. Descriptive Statistic - ICT_GHS_Ind_md.csv
# ==================================================

import os
import pandas as pd

# ==========================================
# 1. DEFINE SHARED WORKSPACE PATHS
# ==========================================
data_dir = '/content/drive/MyDrive/DS_IsTheWorldReadyForTheNextPandemic/DATASET TO USE/'
target_file = "ICT_GHS_Ind_md.csv"
path_target = os.path.join(data_dir, target_file)

print(f"Loading {target_file} for Descriptive Analysis...")

# ==========================================
# 2. READ TARGET DATASET
# ==========================================
if not os.path.exists(path_target):
    raise FileNotFoundError(f"❌ Error: Could not find '{target_file}' in the directory.")

df = pd.read_csv(path_target)

# ==========================================
# 3. RUN UNIFIED DESCRIPTIVE STATISTICS
# ==========================================
print("=" * 75)
print(f"       DESCRIPTIVE STATISTICS SUMMARY: {target_file}         ")
print("=" * 75)

# Configuring display variables so Colab renders the entire width of the summary table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# include='all' builds the comprehensive statistics profile for both numeric and text categories
unified_summary = df.describe(include='all')

print(unified_summary)
print("=" * 75)




Loading ICT_GHS_Ind_md.csv for Descriptive Analysis...
       DESCRIPTIVE STATISTICS SUMMARY: ICT_GHS_Ind_md.csv         
             Entity Code         Year  Landline phone subscriptions  Landline Internet subscriptions  Mobile phone subscriptions  Internet users location_key country_code country_name  subregion1_code  subregion1_name  subregion2_code  subregion2_name  aggregation_level      Country  OVERALL SCORE  1) PREVENTION OF THE EMERGENCE OR RELEASE OF PATHOGENS  1.1) Antimicrobial resistance (AMR)  1.1.1) AMR surveillance, detection and reporting  1.1.1a) National plan for AMR priority pathogens  1.1.1b) Capacity of national lab/lab system to test for AMR priority pathogens  1.1.1c) National environmental surveillance for AMR residues/organisms  1.1.2) Antimicrobial control  1.1.2a) National law(s) requiring prescription for antibiotic use (humans)  1.1.2b) National law(s) requiring prescription for antibiotic use (animals)  1.2) Zoonotic disease  1.2.1) National planning fo